In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

DATA_DIR = Path("../data")
TWCS_PATH = DATA_DIR / "twcs" / "twcs.csv"
SAMPLE_PATH = DATA_DIR / "sample.csv"

print(f"Target dataset exists: {TWCS_PATH.exists()}")

Target dataset exists: True


## 1. Schema Inspection & Data Integrity

In [2]:
# Load schema and first 10 rows
df_sample = pd.read_csv(SAMPLE_PATH if SAMPLE_PATH.exists() else TWCS_PATH, nrows=1000)
print("Shape:", df_sample.shape)
print("\nData Types and Non-Null Counts:")
print(df_sample.info())
df_sample.head(5)

Shape: (93, 7)

Data Types and Non-Null Counts:
<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tweet_id                 93 non-null     int64  
 1   author_id                93 non-null     str    
 2   inbound                  93 non-null     bool   
 3   created_at               93 non-null     str    
 4   text                     93 non-null     str    
 5   response_tweet_id        65 non-null     str    
 6   in_response_to_tweet_id  68 non-null     float64
dtypes: bool(1), float64(1), int64(1), str(4)
memory usage: 4.6 KB
None


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,119237,105834,True,Wed Oct 11 06:55:44 +0000 2017,@AppleSupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡,119236,NaN
1,119238,ChaseSupport,False,Wed Oct 11 13:25:49 +0000 2017,"@105835 Your business means a lot to us. Please DM your name, zip code and additional details about your concern. ^R...",NaN,119239.0
2,119239,105835,True,Wed Oct 11 13:00:09 +0000 2017,@76328 I really hope you all change but I'm sure you won't! Because you don't have to!,119238,NaN
3,119240,VirginTrains,False,Tue Oct 10 15:16:08 +0000 2017,"@105836 LiveChat is online at the moment - https://t.co/SY94VtU8Kq or contact 03331 031 031 option 1, 4, 3 (Leave a ...",119241,119242.0
4,119241,105836,True,Tue Oct 10 15:17:21 +0000 2017,@VirginTrains see attached error message. I've tried leaving a voicemail several times in the past week https://t.co...,119243,119240.0


## 2. Inbound (Customer) vs. Outbound (Brand) Distinction

- `inbound == True`: Customer tweet asking for support (anonymized numeric author_id, e.g. `105834`).
- `inbound == False`: Brand support agent reply (official corporate handle, e.g. `AppleSupport`).

In [3]:
print("Inbound vs Outbound breakdown in sample:")
print(df_sample['inbound'].value_counts(normalize=True))

print("\nSample customer authors (inbound=True):")
print(df_sample[df_sample['inbound'] == True]['author_id'].head(5).tolist())

print("\nSample brand authors (inbound=False):")
print(df_sample[df_sample['inbound'] == False]['author_id'].value_counts().head(5))

Inbound vs Outbound breakdown in sample:
inbound
True     0.526882
False    0.473118
Name: proportion, dtype: float64

Sample customer authors (inbound=True):
['105834', '105835', '105836', '105836', '105836']

Sample brand authors (inbound=False):
author_id
AppleSupport       13
SpotifyCares        8
Tesco               8
VirginTrains        4
British_Airways     3
Name: count, dtype: int64


## 3. Conversation Threading & Pair Reconstruction

How conversations are linked:
- `in_response_to_tweet_id`: Points to the immediate parent tweet.
- A customer complaint has `inbound=True`.
- The brand's direct resolution has `inbound=False` and `in_response_to_tweet_id == customer.tweet_id`.
- Joining these two tables produces clean (Customer Query -> Brand Resolution) pairs.

In [4]:
# Reconstruct Customer -> Support pairs
brand_replies = df_sample[(df_sample['inbound'] == False) & (df_sample['in_response_to_tweet_id'].notna())].copy()
brand_replies['in_response_to_tweet_id'] = brand_replies['in_response_to_tweet_id'].astype(int)
customer_tweets = df_sample[df_sample['inbound'] == True]

pairs = pd.merge(
    brand_replies,
    customer_tweets,
    left_on='in_response_to_tweet_id',
    right_on='tweet_id',
    suffixes=('_brand', '_customer')
)
print(f"Reconstructed {len(pairs)} direct pairs in sample")
for i, row in pairs.head(3).iterrows():
    print(f"\n--- PAIR {i+1} [{row['author_id_brand']}] ---")
    print(f"Customer: {row['text_customer']}")
    print(f"Brand:    {row['text_brand']}")

Reconstructed 42 direct pairs in sample

--- PAIR 1 [ChaseSupport] ---
Customer: @76328 I really hope you all change but I'm sure you won't! Because you don't have to!
Brand:    @105835 Your business means a lot to us. Please DM your name, zip code and additional details about your concern. ^RR https://t.co/znUu1VJn9r

--- PAIR 2 [VirginTrains] ---
Customer: @VirginTrains I still haven't heard &amp; the number I'm directed to by phone is a dead end &amp; the live chat doesn't work. Can someone call me?
Brand:    @105836 LiveChat is online at the moment - https://t.co/SY94VtU8Kq or contact 03331 031 031 option 1, 4, 3 (Leave a message) to request a call back

--- PAIR 3 [VirginTrains] ---
Customer: @VirginTrains see attached error message. I've tried leaving a voicemail several times in the past week https://t.co/NxVZjlYx1k
Brand:    @105836 Have you tried from another device, Miriam ^MM


## 4. Brand Comparison & Recommendation Summary

Load pre-computed brand statistics generated by `src/explore_dataset.py`.

In [5]:
stats_path = DATA_DIR / "analysis" / "brand_statistics.csv"
if stats_path.exists():
    brand_stats = pd.read_csv(stats_path)
    display(brand_stats.head(10))
else:
    print("Run `python src/explore_dataset.py` to generate the complete brand statistics.")

Run `python src/explore_dataset.py` to generate the complete brand statistics.
